# Libraries 

In [2]:
import html
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

# Constants and Parameters

In [46]:
base_url = "https://www.daad.de/en/study-and-research-in-germany/courses-of-study-in-germany/all-study-programmes-in-germany/"

In [4]:
#Parameter Codes
degree_program_code = "w" #Graduate Courses Code
subject_group = "1-226" #Computer Science Code
degree_type = "37" #Masters Code
institution_type = "2" #University of Applied Sciences Code
course_language = "2" #English Code

In [5]:
#Parameters
parameters = {
    "hec-degreeProgrammeType":degree_program_code,
    "hec-subjectGroup":subject_group,
    "hec-degreeType":degree_type,
    "hec-institutionType":institution_type,
    "hec-teachingLanguage":course_language
}

# Functions

In [25]:
def get_query_info(soup, tag:str, class_name=None, id_name=None):
    
    query_list = []
    
    if class_name!=None:
        for info in soup.find_all(tag, class_=class_name):
            query_list.append(info.text.strip())
    elif id_name!=None:
        for info in soup.findAll(tag, {"id":id_name}):
            query_list.append(info.text.strip())
    else:
        for info in soup.findAll(tag):
            query_list.append(info.text.strip())
        
    return query_list

In [9]:
def try_info_list(info_list, index):
    if 0<=index<len(info_list):
        return info_list[index]
    else:
        return np.nan

In [41]:
def get_daad_info(base_url:str, page=1):
    query_path = "?"
    for k,v in parameters.items():
        query_path += f"{k}={v}&"
    
    if page>1:
        query_path += f"hec-p={page}"
    else:
        #Remove the last '&' for the 1st page
        query_path = query_path[:-1]
        
    url = base_url + query_path
    response = requests.get(url)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, "html.parser")
    
    #All Universities Names
    universities_names = get_query_info(soup, "h3")
    universities_names = universities_names[2:-8]
    
    #All DAAD info DataFrame
    df_daad = pd.DataFrame()
    
    for i in range(len(universities_names)):
        university_href = str(soup.find_all("a", class_ = "qa-more-link")[i]).split(" ")[9]
        new_path = university_href.split('"')[1]
        university_url = "https://www.daad.de" + new_path
        university_url = html.unescape(university_url)

        response2 = requests.get(university_url)

        soup2 = BeautifulSoup(response2.content, "html.parser")

        #Degree Name
        degree_name = str(soup2.find("h3")).split(">")[1][:-4]

        #Info List 1
        ##Type of Studies, Semesters, Locations
        info_list = get_query_info(soup2, "h4")
        type_of_studies = try_info_list(info_list, 1)
        semesters = try_info_list(info_list, 3)
        locations = try_info_list(info_list, 4)
    
        #Info List 2
        ##Admission semester, learnings, Admission Requirements
        info_list = get_query_info(soup2, "p", class_name="js-dynamic-content")
        admission_semester = try_info_list(info_list, 1)
        learnings = try_info_list(info_list, 2)
        admission_requirements = try_info_list(info_list, 5)
        
        #Info List 3
        ##Tuition Fees
        info_list = get_query_info(soup2, "div", id_name="hsk-detail-fees")
        tuition_fees = "".join(info_list)
        
        #Info List 4
        #Course Language
        info_list = get_query_info(soup2, "div", id_name="hsk-detail-languages")
        course_language = "".join(info_list)

        university_all_info = {
            "Varsity Name": [universities_names[i]],
            "Degree Name": [degree_name],
            "Type of Studies": [type_of_studies],
            "No. of Semesters": [semesters],
            "Varsity Locations":[locations],
            "Admission": [admission_semester],
            "Focus": [learnings],
            "Admission Requirements": [admission_requirements],
            "Tuition Fees": [tuition_fees],
            "Language": [course_language],
            "Link": [str(university_url)]
        }

        df_temp = pd.DataFrame(university_all_info)
        df_daad = pd.concat([df_daad, df_temp])
        
    return df_daad

# Testing 

In [55]:
df_daad_all = pd.DataFrame()
for page in range(5):
    df_t = get_daad_info(base_url,page)
    if len(df_t)==0:
        break
    df_daad_all = pd.concat([df_daad_all, df_t])
df_daad_all

,Varsity Name,Degree Name,Type of Studies,No. of Semesters,Varsity Locations,Admission,Focus,Admission Requirements,Tuition Fees,Language,Link
0,International School of Management,Applied Business Data Science,"part time degree programmes for professionals,...",4 semesters,"Dortmund, Frankfurt am Main, München, Hamburg,...",Admission in all quarters,"Business Informatics, Statistics, Business Int...","50 ECTS points in the field of Economics, of w...","Tuition fee\n Fees12,450.00 EUR / totalFees...",Languages of instruction\n Main languageEng...,https://www.daad.de/en/study-and-research-in-g...
0,SRH University of Applied Sciences Heidelberg,Applied Computer Science,"international course, full time",24 months,Heidelberg,Summer and Winter Semester,"Business Computing, Multimedia Development",Students who meet the following requirements c...,Tuition fee\n Fees750.00 EUR / MonthFees (L...,Languages of instruction\n Main languageEng...,https://www.daad.de/en/study-and-research-in-g...
0,Hochschule Schmalkalden,Applied Computer Science,full time,4 semesters,Schmalkalden,Winter Semester only,open admission,English,Tuition fee\n Fees130.00 EUR / semesterFees...,Languages of instruction\n Main languageEng...,https://www.daad.de/en/study-and-research-in-g...
0,SRH University of Applied Sciences Heidelberg,Applied Data Science &amp; Analytics,"international course, full time",4 semesters,Heidelberg,Summer and Winter Semester,"Data Management, Big Data, Data Protection, Da...",Selection procedure / qualifying examination,Tuition fee\n Fees790.00 EUR / MonthFees (L...,Languages of instruction\n Main languageEng...,https://www.daad.de/en/study-and-research-in-g...
0,Mittweida University of Applied Sciences,Applied Mathematics for Network and Data Sciences,"international course, full time",4 semesters,Mittweida,Winter Semester only,"Cryptography, Mathematical Network Analysis, C...",Missing documents can be submitted by 21.07. o...,Tuition fee\n The university may charge tui...,Languages of instruction\n Main languageEng...,https://www.daad.de/en/study-and-research-in-g...
...,...,...,...,...,...,...,...,...,...,...,...
0,Stuttgart University of Applied Sciences,Software Technology,"international course, part time, full time",3 semesters,Stuttgart,Summer and Winter Semester,Future decision-makers and project managers in...,"Diplom or Bachelor's degree in Mathematics, Co...",,Languages of instruction\n Main languageEng...,https://www.daad.de/en/study-and-research-in-g...
0,University of Applied Sciences Neu-Ulm,Strategic Information Management,"international course, full time",3 semesters,Neu-Ulm,Winter Semester only,"Enterprise Information Systems, Business Intel...",- completed university degree 210 ECTS grade 2...,,Languages of instruction\n Main languageEng...,https://www.daad.de/en/study-and-research-in-g...
0,Rhine-Waal University of Applied Sciences,Usability Engineering,"international course, full time",3 semesters,Kamp-Lintfort,Summer and Winter Semester,"Usability, User Experience, User Research, Usa...",Enquire at the university,Tuition fee\n FeesFees (Link),Languages of instruction\n Main languageEng...,https://www.daad.de/en/study-and-research-in-g...
0,Technische Hochschule Ingolstadt,User Experience Design,"dual system, full time",3 semesters,Ingolstadt,Summer and Winter Semester,selection by the HEI,NaN,,Languages of instruction\n Main languageEng...,https://www.daad.de/en/study-and-research-in-g...


# Convert to `.csv` 

In [60]:
df_daad_all.to_csv("DAAD_University_info.csv", encoding='utf-8')

In [61]:
print("Successful!")

Successful!
